In [ ]:
# ANÁLISIS ESTADÍSTICO EMPRESARIAL — PASO A PASO EJECUTABLE
# Dataset : 1.000 Empresas más grandes de Colombia (Supersociedades 2018)
# Autor   : Sebastian montoya echeverry_BIanalist
#objetivo : Determinar cuales son los sectores economicos mas rentables del pais, conocer su beneficio y crecimiento entre el ano 2017 y 2018 para asi poder determinar las proyecciones futuras de cada empresa en cada sector.

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  #visualizar las graficas en PNG        
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore') #evitar advertencias que no son errores

In [3]:
df=pd.read_csv("1000_Empresas_mas_grandes_del_pais.csv")
print("columnas_originales:")
print(df.columns.tolist())

columnas_originales:
['No.', 'NIT', 'RAZON SOCIAL', 'SUPERVISOR', 'REGIÓN', 'DEPARTAMENTO DOMICILIO', 'CIUDAD DOMICILIO', 'CIIU', 'MACROSECTOR', 'INGRESOS OPERACIONALES\n2018*', 'GANANCIA (PERDIDA) 2018', 'TOTAL ACTIVOS 2018', 'TOTAL PASIVOS 2018', 'TOTAL PATRIMONIO 2018', 'INGRESOS OPERACIONALES\n2017*', 'GANANCIA (PERDIDA) 2017', 'TOTAL ACTIVOS 2017', 'TOTAL PASIVOS 2017', 'TOTAL PATRIMONIO 2017', 'GRUPO EN NIIF']


In [4]:
COLUMNAS = [
    'No','NIT','RAZON_SOCIAL','SUPERVISOR','REGION','DEPTO','CIUDAD',
    'CIIU','MACROSECTOR','ING_OP_2018','GANANCIA_2018','ACT_2018',
    'PAS_2018','PAT_2018','ING_OP_2017','GANANCIA_2017','ACT_2017',
    'PAS_2017','PAT_2017','GRUPO_NIIF'
]
col_num=['ING_OP_2018','GANANCIA_2018','ACT_2018','PAS_2018','PAT_2018','ING_OP_2017','GANANCIA_2017','ACT_2017',
    'PAS_2017','PAT_2017']
semilla=42

In [5]:
df.columns=COLUMNAS
print(f"✅  Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"    Período: 2017-2018 | Fuente: Supersociedades Colombia")
print(df.head(3).to_string())

✅  Dataset cargado: 1000 filas × 20 columnas
    Período: 2017-2018 | Fuente: Supersociedades Colombia
   No        NIT                   RAZON_SOCIAL       SUPERVISOR                 REGION        DEPTO                   CIUDAD                                                                                           CIIU           MACROSECTOR   ING_OP_2018  GANANCIA_2018      ACT_2018      PAS_2018      PAT_2018   ING_OP_2017  GANANCIA_2017      ACT_2017      PAS_2017      PAT_2017           GRUPO_NIIF
0   1  899999068                  ECOPETROL S.A  SUPERFINANCIERA  Bogotá - Cundinamarca  BOGOTA D.C.  BOGOTA-D.C.-BOGOTA D.C.                                                           B0610 - Extracción de petróleo crudo  MINERO-HIDROCARBUROS  6.257985e+10   1.155640e+10  1.137618e+11  5.654822e+10  5.721361e+10  4.968708e+10   6.620412e+09  1.075490e+11  5.965040e+10  4.789863e+10  NIIF PLENAS-GRUPO 1
1   2  830095213       ORGANIZACIÓN TERPEL S.A.  SUPERFINANCIERA  Bogotá - Cundinamar

In [6]:
print("\n" + "="*65)
print("LIMPIEZA · ANÁLISIS DE CALIDAD DETALLADO")
print("="*65)

# 1. NULOS POR COLUMNA
print("\n--- NULOS POR COLUMNA ---")
nulos_col = df.isnull().sum()
print(nulos_col[nulos_col > 0] if nulos_col.sum() > 0 else "✅ Sin valores nulos")

# 2. DUPLICADOS
print("\n--- DUPLICADOS ---")
print(f"✅ Filas duplicadas: {df.duplicated().sum()}")
print(f"✅ NITs duplicados : {df['NIT'].duplicated().sum()}")

# 3. CONSISTENCIA COLUMNAS CATEGÓRICAS
print("\n--- VALORES ÚNICOS COLUMNAS CATEGÓRICAS ---")
for col in ['MACROSECTOR','REGION','SUPERVISOR','GRUPO_NIIF']:
    print(f"\n{col}:")
    print(df[col].value_counts().to_string())

# 4. VALORES IMPOSIBLES — USANDO COLS_NUM
print("\n--- VALORES IMPOSIBLES EN COLUMNAS NUMÉRICAS ---")
for col in col_num:
    ceros    = (df[col] == 0).sum()
    negativos = (df[col] < 0).sum()
    print(f"{col:25s} → ceros: {ceros:3d} | negativos: {negativos:3d}")

# 5. CONSISTENCIA CONTABLE
print("\n--- CONSISTENCIA CONTABLE ---")
print(f"Activos ≠ Pasivos + Patrimonio 2018: "
      f"{((df['ACT_2018'] - (df['PAS_2018'] + df['PAT_2018'])).abs() > 1000).sum()} empresas")
print(f"Activos ≠ Pasivos + Patrimonio 2017: "
      f"{((df['ACT_2017'] - (df['PAS_2017'] + df['PAT_2017'])).abs() > 1000).sum()} empresas")


LIMPIEZA · ANÁLISIS DE CALIDAD DETALLADO

--- NULOS POR COLUMNA ---
✅ Sin valores nulos

--- DUPLICADOS ---
✅ Filas duplicadas: 0
✅ NITs duplicados : 0

--- VALORES ÚNICOS COLUMNAS CATEGÓRICAS ---

MACROSECTOR:
MACROSECTOR
MANUFACTURA             328
COMERCIO                303
SERVICIOS               225
CONSTRUCCIÓN             66
MINERO-HIDROCARBUROS     54
AGROPECUARIO             24

REGION:
REGION
Bogotá - Cundinamarca    555
Antioquia                161
Costa Pacífica           127
Costa Atlántica           98
Centro - Oriente          30
Eje Cafetero              22
Otros                      7

SUPERVISOR:
SUPERVISOR
SUPERSOCIEDADES    896
SUPERSALUD          42
SUPERFINANCIERA     38
SUPERSERVICIOS      16
SUPERVIGILANCIA      8

GRUPO_NIIF:
GRUPO_NIIF
NIIF PLENAS-GRUPO 1            760
NIIF PYMES-GRUPO 2             232
REGIMEN R 414 de 2014 - CGN      8

--- VALORES IMPOSIBLES EN COLUMNAS NUMÉRICAS ---
ING_OP_2018               → ceros:   0 | negativos:   0
GANANCIA_2018  

#**Despues de la evaluacion no debemos de realizar ningun tipo de limpieza ya que los datos son congruentes y no hay errores atipicos**

In [7]:
print("\n" + "="*65)
print("PASO 2 · AUDITORÍA DE CALIDAD DE DATOS")
print("="*65)

duplicados      = df.duplicated().sum()
nulos_total     = df.isnull().sum().sum()
pat_negativo    = (df['PAT_2018'] < 0).sum()
ganancia_neg    = (df['GANANCIA_2018'] < 0).sum()
endeud_mayor1   = (df['PAS_2018'] > df['ACT_2018']).sum()

print(f"  Duplicados          : {duplicados}")
print(f"  Valores nulos       : {nulos_total}")
print(f"  Patrimonio negativo : {pat_negativo}  empresas  "
      f"({pat_negativo/len(df)*100:.1f}%) → técnicamente insolventes")
print(f"  Ganancia negativa   : {ganancia_neg} empresas  "
      f"({ganancia_neg/len(df)*100:.1f}%) → operaron con pérdida")
print(f"  Pasivos > Activos   : {endeud_mayor1}  empresas  "
      f"→ patrimonio neto negativo (bancarrota técnica)")


PASO 2 · AUDITORÍA DE CALIDAD DE DATOS
  Duplicados          : 0
  Valores nulos       : 0
  Patrimonio negativo : 48  empresas  (4.8%) → técnicamente insolventes
  Ganancia negativa   : 193 empresas  (19.3%) → operaron con pérdida
  Pasivos > Activos   : 48  empresas  → patrimonio neto negativo (bancarrota técnica)


In [8]:
# Detalle empresas con patrimonio negativo
print("\n  Top 5 empresas con mayor insolvencia técnica:")
insolventes = df[df['PAT_2018'] < 0][['RAZON_SOCIAL','PAT_2018','MACROSECTOR']]
print(insolventes.nsmallest(5,'PAT_2018').to_string(index=False))



  Top 5 empresas con mayor insolvencia técnica:
      RAZON_SOCIAL      PAT_2018 MACROSECTOR
SALUDVIDA S.A. EPS -7.303930e+08   SERVICIOS
   SAVIA SALUD EPS -7.107629e+08   SERVICIOS
      ASMET S.A.S. -6.991683e+08   SERVICIOS
    COOMEVA EPS SA -4.335343e+08   SERVICIOS
      EMSSANAR ESS -4.209866e+08   SERVICIOS


**Generamos estadistica descriptiva para conocer como estan distribuidos los datos a nivel global**

In [9]:
print("\n" + "="*65)
print("PASO 3 · ESTADÍSTICA DESCRIPTIVA")
print("="*65)

# 1. RESUMEN GENERAL
print("\n--- RESUMEN GENERAL ---")
print(df[col_num].describe())

# 2. MEDIDAS DE TENDENCIA CENTRAL
print("\n--- MEDIDAS DE TENDENCIA CENTRAL ---")
for col in col_num:
    media   = df[col].mean()
    mediana = df[col].median()
    moda    = df[col].mode()[0]
    print(f"{col:25s} media: {media:>20,.0f} | mediana: {mediana:>20,.0f} | moda: {moda:>20,.0f}")

# 3. MEDIDAS DE DISPERSIÓN
print("\n--- MEDIDAS DE DISPERSIÓN ---")
for col in col_num:
    std    = df[col].std()
    rango  = df[col].max() - df[col].min()
    cv     = (std / df[col].mean()) * 100
    print(f"{col:25s} std: {std:>20,.0f} | rango: {rango:>20,.0f} | CV%: {cv:>8.1f}%")

# 4. MEDIDAS DE POSICIÓN
print("\n--- CUARTILES ---")
print(df[col_num].quantile([0.25, 0.50, 0.75]))

# 5. DIAGNÓSTICO DE SESGO
print("\n--- DIAGNÓSTICO DE SESGO ---")
for col in col_num:
    media   = df[col].mean()
    mediana = df[col].median()
    #dividimos media en media, si la mediana es 0 da error, si es 0 debe poner infinito, si el resultado es muy diferente de 1 hay sesgo en los datos
    ratio   = abs(media / mediana) if mediana != 0 else float('inf')
    #cieficiente de asimetria, colas o centralizado
    sesgo   = df[col].skew()
    nivel   = "⚠ ALTO"   if ratio > 3 else \
              "△ MEDIO"  if ratio > 1.5 else "BAJO"
    print(f"{col:25s} ratio media/mediana: {ratio:5.1f}x | sesgo: {sesgo:7.2f} | {nivel}")


PASO 3 · ESTADÍSTICA DESCRIPTIVA

--- RESUMEN GENERAL ---
        ING_OP_2018  GANANCIA_2018      ACT_2018      PAS_2018      PAT_2018  \
count  1.000000e+03   1.000000e+03  1.000000e+03  1.000000e+03  1.000000e+03   
mean   6.799250e+08   6.855500e+07  9.862732e+08  4.598472e+08  5.264260e+08   
std    2.274348e+09   4.498809e+08  4.505014e+09  2.154427e+09  2.535467e+09   
min    1.343546e+08  -8.024497e+08  7.329964e+06  4.270000e+04 -7.303930e+08   
25%    1.770936e+08   1.013276e+06  1.196560e+08  6.635366e+07  3.197481e+07   
50%    2.728692e+08   6.456776e+06  2.409461e+08  1.312229e+08  8.352536e+07   
75%    5.645334e+08   2.423737e+07  5.937477e+08  3.185258e+08  2.496863e+08   
max    6.257985e+10   1.155640e+10  1.137618e+11  5.654822e+10  5.721361e+10   

        ING_OP_2017  GANANCIA_2017      ACT_2017      PAS_2017      PAT_2017  
count  1.000000e+03   1.000000e+03  1.000000e+03  1.000000e+03  1.000000e+03  
mean   5.885920e+08   4.369164e+07  9.055083e+08  4.069496e+08

#**En este paso podemos inferir:**
-la media no nos sirve como medida representativa de los datos
-esto se explica por la gran diferencia entre el 50% (mediana) con respecto al promedio, las empresas grande jalan la media hacia arriba.
-todas las variables tienen un sesgo positivo lo que quiere decir:
--las ganancias esta extremadamente concentrada en las empresas grandes
--los activos tambien estan concentrados en las empresas grandes donde una estructura empresarial colombiana muy desigual
--la mayor parte de las deduas empresariales colombianas estan concetradas en las empresas mas grandes
--el patrimonio es la variable mas desigual, pat_2018 6.3x sesgo 14.11 las empresas mas grandes del pais concentran casi todo el capital del pais

#**PRUEBAS DE NORMALIDAD**
--Esto con el fin de determinar si los datos siguen una distribucion normal para determinar como manejar los datos sin de forma parametrica (datos normales) o de forma no parametrica(datos que no siguen distribucion normal)

In [10]:

import math

print("\n" + "="*65)
print("TAMAÑO DE MUESTRA — CÁLCULO ESTADÍSTICO")
print("="*65)


TAMAÑO DE MUESTRA — CÁLCULO ESTADÍSTICO


In [11]:
N=len(df)
Z=1.96
e=0.05
prop=0.5 #maxima heterogeneidad, los datos tienen mucha diferencia entre si, con datos mas homogeneos puede variar ese indice hasta 0.1

In [12]:
#formula tamano muestra
n_base=(Z**2*prop*(1-prop))/e**2
n_ajustada=n_base/(1+(n_base/N))
muestra= math.ceil(n_ajustada) #redondeo hacia arriba

print(f"\n  Población total  (N) : {N} empresas")
print(f"  Nivel de confianza   : {Z} → {int(Z*100/1.96*95/100+0.5)}%")
print(f"  Margen de error  (e) : {e*100:.0f}%")
print(f"  Proporción       (p) : {prop}")
print(f"\n  Muestra base         : {math.ceil(n_base)}")
print(f"  Muestra ajustada     : {muestra}")

print(f"""
  INTERPRETACIÓN:
  Para una población de {N} empresas con 95% de confianza
  y 5% de margen de error necesitamos mínimo {muestra} empresas
  en cada prueba de normalidad.
  
  El código usará muestra = {muestra} en todas las pruebas.
""")


  Población total  (N) : 1000 empresas
  Nivel de confianza   : 1.96 → 95%
  Margen de error  (e) : 5%
  Proporción       (p) : 0.5

  Muestra base         : 385
  Muestra ajustada     : 278

  INTERPRETACIÓN:
  Para una población de 1000 empresas con 95% de confianza
  y 5% de margen de error necesitamos mínimo 278 empresas
  en cada prueba de normalidad.

  El código usará muestra = 278 en todas las pruebas.



In [13]:
print("\n" + "="*65)
print("PASO 4 · PRUEBAS DE NORMALIDAD")
print("="*65)

# ── PRUEBA 1: SHAPIRO-WILK ───────────────────────────────────────
print(f"\n--- SHAPIRO-WILK (muestra n={muestra} por variable) ---")
resultados_norm = {}
for col in col_num:
    n_muestra    = df[col].sample(muestra, random_state=semilla)
    w, p       = stats.shapiro(n_muestra.values)
    es_normal  = p >= 0.05
    resultados_norm[col] = es_normal
    icono      = "Normal" if es_normal else "NO normal"
    print(f"  {col:25s}  W={w:.4f}  p={p:.2e}  → {icono}")

# ── PRUEBA 2: KOLMOGOROV-SMIRNOV ────────────────────────────────
tam_muestra=len(n_muestra)
print(f"\n--- KOLMOGOROV-SMIRNOV (muestra n={tam_muestra} por variable) ---")
resultados_ks = {}
for col in col_num:
    n_muestra       = df[col].sample(tam_muestra, random_state=semilla)
    media_m       = n_muestra.mean()
    std_m         = n_muestra.std()
    stat, p       = stats.kstest(n_muestra.values, 'norm', args=(media_m, std_m))
    es_normal_ks  = p >= 0.05
    resultados_ks[col] = es_normal_ks
    icono         = "Normal" if es_normal_ks else " NO normal"
    print(f"  {col:25s}  D={stat:.4f}  p={p:.2e}  → {icono}")

# ── PRUEBA 3: SESGO Y CURTOSIS ───────────────────────────────────
print("\n--- SESGO Y CURTOSIS ---")
print(f"  {'Variable':25s}  {'Sesgo':>10}  {'Curtosis':>10}  {'Diagnóstico'}")
for col in col_num:
    sesgo    = df[col].skew()
    curtosis = df[col].kurtosis()
    diag     = "Normal" if abs(sesgo) < 0.5 and abs(curtosis) < 1 else "NO normal"
    print(f"  {col:25s}  {sesgo:>10.3f}  {curtosis:>10.3f}  {diag}")

# ── CONCLUSIÓN AUTOMÁTICA ────────────────────────────────────────
print("\n--- CONCLUSIÓN ---")
normales    = [col for col, v in resultados_norm.items() if v]
no_normales = [col for col, v in resultados_norm.items() if not v]

if len(no_normales) == len(col_num):
    print("   Ninguna variable sigue distribución normal")
elif len(normales) == len(col_num):
    print("Todas las variables siguen distribución normal")
else:
    print(f"  ⚠ Variables normales    : {normales}")
    print(f"  ⚠ Variables NO normales : {no_normales}")

print("\n  IMPLICACIONES PARA EL ANÁLISIS:")
print("  → Medida central recomendada  : MEDIANA (no media)")
print("  → Comparación entre grupos    : Kruskal-Wallis (no ANOVA)")
print("  → Correlación recomendada     : Spearman (no Pearson)")
print("  → Detección de outliers       : IQR (no Z-score)")



PASO 4 · PRUEBAS DE NORMALIDAD

--- SHAPIRO-WILK (muestra n=278 por variable) ---
  ING_OP_2018                W=0.3712  p=6.84e-30  → NO normal
  GANANCIA_2018              W=0.2852  p=2.16e-31  → NO normal
  ACT_2018                   W=0.2775  p=1.61e-31  → NO normal
  PAS_2018                   W=0.2547  p=6.84e-32  → NO normal
  PAT_2018                   W=0.2375  p=3.64e-32  → NO normal
  ING_OP_2017                W=0.3863  p=1.31e-29  → NO normal
  GANANCIA_2017              W=0.3663  p=5.56e-30  → NO normal
  ACT_2017                   W=0.2649  p=1.00e-31  → NO normal
  PAS_2017                   W=0.2488  p=5.51e-32  → NO normal
  PAT_2017                   W=0.2312  p=2.91e-32  → NO normal

--- KOLMOGOROV-SMIRNOV (muestra n=278 por variable) ---
  ING_OP_2018                D=0.3475  p=1.67e-30  →  NO normal
  GANANCIA_2018              D=0.3508  p=4.23e-31  →  NO normal
  ACT_2018                   D=0.3801  p=1.29e-36  →  NO normal
  PAS_2018                   D=0.3752 

**Resultados de las pruebas de normalidad**
--PRUEBA SHAPIRO-WILK: El valor W se aleja de 1 lo cual nos dice que los datos no corresponden a una distribucion normal
--Los p_value son muy cercanos a 0, menores a 0.5 lo que quiere decir que rechamos la h0 con 95% de confianza hay suficiente evidencia estadistica para decir que los datos no corresponden a una distribucion normal
--PRUEBA KOLMOGOROV-SMIRNOV: El valor D mide la distancia max entre la distribucion normal y la real
los valores estan entre 0.31 y 0.39 muy alejado de 0 lo cual no obedece a una distribucion normal

**PASOS A SEGUIR DADO LOS RESULTADOS DE LAS PRUEBAS**
-al no tener datos que obedezcan a una distribucion normal debemos:
-usamos kruskal-wallis en vez de ANOVA para compracion de sectores
-Evaluamos correlacion de Spearman
-Manejo de datos extremos IQR en vez de Z-score
-comparacion entre dos grupos Mann-Whitney en vez de T-test

In [14]:
print("\n" + "="*65)
print("PASO 5 · DETECCIÓN Y TRATAMIENTO DE OUTLIERS")
print("="*65)

# ── PARTE 1: DETECCIÓN IQR TODAS LAS VARIABLES ──────────────────
print("\n--- DETECCIÓN OUTLIERS POR VARIABLE (Método IQR) ---")
print(f"  {'Variable':25s}  {'N outliers':>10}  {'%':>6}  {'Lím inferior':>20}  {'Lím superior':>20}")

resumen_outliers = []
for col in col_num:
    Q1      = df[col].quantile(0.25)
    Q3      = df[col].quantile(0.75)
    IQR     = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    mask    = (df[col] < lim_inf) | (df[col] > lim_sup)
    n_out   = mask.sum()
    resumen_outliers.append({
        'variable'  : col,
        'Q1'        : Q1,
        'Q3'        : Q3,
        'IQR'       : IQR,
        'lim_inf'   : lim_inf,
        'lim_sup'   : lim_sup,
        'n_outliers': n_out,
        'pct'       : n_out / len(df) * 100
    })
    print(f"  {col:25s}  {n_out:>10}  {n_out/len(df)*100:>5.1f}%  "
          f"{lim_inf:>20,.0f}  {lim_sup:>20,.0f}")

# ── PARTE 2: COMPARACIÓN OUTLIERS 2017 vs 2018 ──────────────────
print("\n--- COMPARACIÓN OUTLIERS 2017 vs 2018 ---")
pares = [
    ('ING_OP_2017',   'ING_OP_2018'),
    ('GANANCIA_2017', 'GANANCIA_2018'),
    ('ACT_2017',      'ACT_2018'),
    ('PAS_2017',      'PAS_2018'),
    ('PAT_2017',      'PAT_2018')
]

print(f"  {'Variable':20s}  {'Outliers 2017':>15}  {'Outliers 2018':>15}  {'Variación':>10}")
for col17, col18 in pares:
    # 2017
    Q1_17   = df[col17].quantile(0.25)
    Q3_17   = df[col17].quantile(0.75)
    IQR_17  = Q3_17 - Q1_17
    n_17    = ((df[col17] < Q1_17 - 1.5*IQR_17) | (df[col17] > Q3_17 + 1.5*IQR_17)).sum()
    #aca identificamos cuantas empresas estan por debajo y por encima de los limites interquartiles asi sabemos el total de outliers
    # 2018
    Q1_18   = df[col18].quantile(0.25)
    Q3_18   = df[col18].quantile(0.75)
    IQR_18  = Q3_18 - Q1_18
    n_18    = ((df[col18] < Q1_18 - 1.5*IQR_18) | (df[col18] > Q3_18 + 1.5*IQR_18)).sum()
    # variacion de la cantidad de empresas que estan por fuera de los limites interquartiles
    variacion = n_18 - n_17
    signo     = "↑ aumentaron" if variacion > 0 else "↓ disminuyeron" if variacion < 0 else "= sin cambio"
    nombre    = col17.replace('_2017','')
    print(f"  {nombre:20s}  {n_17:>15}  {n_18:>15}  {signo} {abs(variacion)}")

# ── PARTE 3: LISTADO DE EMPRESAS OUTLIERS POR VARIABLE ──────────
print("\n--- EMPRESAS OUTLIERS POR VARIABLE ---")
for col in col_num:
    Q1      = df[col].quantile(0.25)
    Q3      = df[col].quantile(0.75)
    IQR     = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    mask    = (df[col] < lim_inf) | (df[col] > lim_sup)
    empresas_out = df[mask][['RAZON_SOCIAL', 'MACROSECTOR', col]].copy()
    empresas_out = empresas_out.sort_values(col, ascending=False)
    print(f"\n  {col} — {mask.sum()} empresas outlier:")
    print(f"  {'Empresa':40s}  {'Sector':20s}  {'Valor':>20}")
    for _, row in empresas_out.head(10).iterrows():
        print(f"  {row['RAZON_SOCIAL'][:40]:40s}  "
              f"{row['MACROSECTOR'][:20]:20s}  "
              f"{row[col]:>20,.0f}")
    if mask.sum() > 10:
        print(f"  ... y {mask.sum()-10} empresas más")

# ── PARTE 4: TRANSFORMACIÓN LOGARÍTMICA ─────────────────────────
print("\n--- TRANSFORMACIÓN LOGARÍTMICA ---")
# Solo columnas que no tienen valores negativos
cols_log = ['ING_OP_2018', 'ACT_2018', 'PAS_2018',
            'ING_OP_2017', 'ACT_2017', 'PAS_2017']

for col in cols_log:
    negativos = (df[col] <= 0).sum()
    if negativos == 0:
        df[col + '_LOG'] = np.log10(df[col])
        print(f"  ✅ {col:25s} → {col}_LOG creada")
    else:
        print(f"  ⚠  {col:25s} → tiene {negativos} valores negativos, no se transforma")

# Ganancia y Patrimonio tienen negativos — transformación especial
# log(x + |min| + 1) para preservar negativos
for col in ['GANANCIA_2018', 'PAT_2018', 'GANANCIA_2017', 'PAT_2017']:
    minimo = df[col].min()
    ajuste = abs(minimo) + 1
    df[col + '_LOG'] = np.log10(df[col] + ajuste)
    print(f"  ✅ {col:25s} → {col}_LOG creada (ajuste por negativos)")

# ── PARTE 5: DECISIÓN DOCUMENTADA ───────────────────────────────
print("\n--- DECISIÓN SOBRE OUTLIERS ---")
print("""
  EMPRESAS GIGANTES (Ecopetrol, EPM, Grupo Sura):
  → CONSERVAR — son reales y representan la economía colombiana
  → USAR columnas LOG para análisis y modelos

  EMPRESAS CON PATRIMONIO NEGATIVO (48 empresas):
  → CONSERVAR — representan insolvencia técnica real
  → DOCUMENTAR como segmento de riesgo

  EMPRESAS CON GANANCIA NEGATIVA (193 empresas):
  → CONSERVAR — operaron con pérdida, dato válido
  → ANALIZAR como grupo separado en análisis de riesgo
""")

print(f"  Columnas LOG creadas: {[c for c in df.columns if '_LOG' in c]}")
print(f"  Shape final del dataset: {df.shape}")


PASO 5 · DETECCIÓN Y TRATAMIENTO DE OUTLIERS

--- DETECCIÓN OUTLIERS POR VARIABLE (Método IQR) ---
  Variable                   N outliers       %          Lím inferior          Lím superior
  ING_OP_2018                       108   10.8%          -404,066,156         1,145,693,122
  GANANCIA_2018                     184   18.4%           -33,822,857            59,073,500
  ACT_2018                          110   11.0%          -591,481,508         1,304,885,182
  PAS_2018                          108   10.8%          -311,904,578           696,784,044
  PAT_2018                          117   11.7%          -294,592,410           576,253,503
  ING_OP_2017                       117   11.7%          -353,813,128         1,008,942,920
  GANANCIA_2017                     208   20.8%           -27,193,468            46,583,414
  ACT_2017                          116   11.6%          -530,709,595         1,168,486,900
  PAS_2017                          101   10.1%          -274,086,490   

In [15]:
df['ING_OP_2017_LOG'] = np.log10(df['ING_OP_2017'].clip(lower=1))

In [16]:
import plotly.graph_objects as go

# ── GRÁFICO 1 — VARIABLES LOG 2017 vs 2018 ──────────────────────
# color_17 = color claro 2017 | color_18 = color oscuro 2018
variables_log = [
    ('ACT',      'Activos',    '#58A6FF', '#1F6FEB'),
    ('ING_OP',   'Ingresos',   '#3FB950', '#196C2E'),
    ('GANANCIA', 'Ganancia',   '#D2A8FF', '#8957E5'),
    ('PAT',      'Patrimonio', '#FFA657', '#D4691E'),
]
#ACT es el prefijo de la columna en el df, asi se construyen automoaticamente los nombre= col_17=f'{prefijo}_2017_LOG'
#Activos es el nombre para el usuario, posteriormente se le asigna un color diferente a cada ano, se asigna por orden
fig1 = go.Figure()
#creamos objeto vacio a configurar
for prefijo, nombre, color_17, color_18 in variables_log:
    col_17 = f'{prefijo}_2017_LOG'
    col_18 = f'{prefijo}_2018_LOG'
#recorremos las tuplas una por una
    # 2017 — color claro, creamos las caracteristicas del grafico
    fig1.add_trace(go.Box(
        y             = df[col_17],
        name          = f"{nombre} 2017",
        marker_color  = color_17,
        boxpoints     = 'outliers',
        pointpos      = 0,
        marker        = dict(size=5, opacity=0.7, color=color_17,
                             line=dict(width=1, color='white')),
        text          = df['RAZON_SOCIAL'],
        hovertemplate = "<b>%{text}</b><br>Valor: %{y:,.2f}<br>Año: 2017<extra></extra>",
        line          = dict(width=2),
        legendgroup   = nombre,
    ))
#add.trace agreewga una caja nueva por cada columna de datos, los datos y vienen de la columna col_17, asignamos el nombre de la leyenda del grafico
#asignamos el color determinado, outliers solo muestra los puntos individuales, centrado, tamano punto, opacidad, color determinado, caracteristicas de la linea
#texto al pasar el cursor en cad punto, definimos informacion concreta al pasar el mouse(text=nombre empresa, valor con 2 decimales separacion de miles, eliminamos informacion que se pone por defecto)
    # 2018 — color oscuro
    fig1.add_trace(go.Box(
        y             = df[col_18],
        name          = f"{nombre} 2018",
        marker_color  = color_18,
        boxpoints     = 'outliers',
        pointpos      = 0,
        marker        = dict(size=5, opacity=0.7, color=color_18,
                             line=dict(width=1, color='white')),
        text          = df['RAZON_SOCIAL'],
        hovertemplate = "<b>%{text}</b><br>Valor: %{y:,.2f}<br>Año: 2018<extra></extra>",
        line          = dict(width=2),
        legendgroup   = nombre,
    ))
#las mismas anotaciones del year 2017 pero para el  2018

##definimos el aspecto  visual del grafico
fig1.update_layout(
    title        = dict(
        text     = "<b>VARIABLES FINANCIERAS LOG — 2017 vs 2018</b><br>"
                   "<sub>Activos, Ingresos, Ganancia y Patrimonio en escala logarítmica</sub>",
        font     = dict(size=18, color='white'), x=0.5
    ), #text- texto b-negrita sub-texto pequeno subtitulo
    paper_bgcolor = '#0D1117',
    plot_bgcolor  = '#161B22',
    font          = dict(color='#E6EDF3', size=12),
    yaxis         = dict(
        gridcolor     = '#30363D',
        zerolinecolor = '#30363D',
        title         = 'log₁₀(Valor COP)'
    ), #paper-color fondo de la figura, plot color fondo sol ode las cajas, font-definimos caracteristicas de todo el texto del grafico
       #yaxis gid coor lineas cuadricula horizontal, zero-color linea del cero, title la etiqueta del eje
    xaxis         = dict(gridcolor='#30363D'),
    legend        = dict(bgcolor='#161B22', bordercolor='#30363D', borderwidth=1),
    height        = 650,
    boxmode       = 'group',
    annotations   = [dict(
        text      = "Pasa el mouse sobre los puntos para ver el nombre de cada empresa",
        xref="paper", yref="paper", x=0.5, y=-0.15,
        showarrow=False, font=dict(size=11, color='#8B949E'), xanchor='center'
    )])
     #boxmode-dice que las cajas van una al lado de la otra, xref, yref coordenadas relativas al tamano de la figura no a los datos

fig1.show()

# ── GRÁFICO 2 — PASIVOS NORMALES 2017 vs 2018 ───────────────────
fig2 = go.Figure()

for col, nombre, color in [('PAS_2017', 'Pasivos 2017', '#F78166'),
                             ('PAS_2018', 'Pasivos 2018', '#B22222')]:
    fig2.add_trace(go.Box(
        y             = df[col],
        name          = nombre,
        marker_color  = color,
        boxpoints     = 'outliers',
        pointpos      = 0,
        marker        = dict(size=5, opacity=0.7, color=color,
                             line=dict(width=1, color='white')),
        text          = df['RAZON_SOCIAL'],
        hovertemplate = "<b>%{text}</b><br>Pasivos: %{y:,.0f} COP<extra></extra>",
        line          = dict(width=2),
    ))

fig2.update_layout(
    title        = dict(
        text     = "<b>PASIVOS — 2017 vs 2018</b><br>"
                   "<sub>Valores en COP — escala original sin transformación</sub>",
        font     = dict(size=18, color='white'), x=0.5
    ),
    paper_bgcolor = '#0D1117',
    plot_bgcolor  = '#161B22',
    font          = dict(color='#E6EDF3', size=12),
    yaxis         = dict(
        gridcolor     = '#30363D',
        zerolinecolor = '#30363D',
        title         = 'Pasivos (COP)'
    ),
    xaxis         = dict(gridcolor='#30363D'),
    legend        = dict(bgcolor='#161B22', bordercolor='#30363D', borderwidth=1),
    height        = 550,
    boxmode       = 'group',
    annotations   = [dict(
        text      = "💡 Pasa el mouse sobre los puntos para ver el nombre de cada empresa",
        xref="paper", yref="paper", x=0.5, y=-0.15,
        showarrow=False, font=dict(size=11, color='#8B949E'), xanchor='center'
    )]
)

fig2.show()

In [19]:
import plotly.graph_objects as go

# ── PASO 1: TOTALES POR SECTOR ───────────────────────────────────
totales = df.groupby('MACROSECTOR').agg(
    ACT_2017 = ('ACT_2017',      'sum'),
    ACT_2018 = ('ACT_2018',      'sum'),
    PAS_2017 = ('PAS_2017',      'sum'),
    PAS_2018 = ('PAS_2018',      'sum'),
    ING_2017 = ('ING_OP_2017',   'sum'),
    ING_2018 = ('ING_OP_2018',   'sum'),
    GAN_2017 = ('GANANCIA_2017', 'sum'),
    GAN_2018 = ('GANANCIA_2018', 'sum'),
    PAT_2017 = ('PAT_2017',      'sum'),
    PAT_2018 = ('PAT_2018',      'sum'),
).reset_index()
#agrupamos las empresas por sector economico, agg agraga la fincion sum, nombre columna, toma datos, operacion
#reset_index=usa 'macrosector' como indice

##PASO 2: TRANSFORMACIÓN LOGARÍTMICA 
for col in totales.columns[1:]:
    minimo = totales[col].min()
    ajuste = abs(minimo) + 1 if minimo <= 0 else 0
    totales[col+'_LOG'] = np.log10(totales[col] + ajuste)

sectores = totales['MACROSECTOR'].tolist()
#recorremos cada columna menos la 0 que text, cogemos el valor mas pequeno detectando si hay negativos
#ajustamos los valores de logaritmo, si el min<0 pasa como log10(1)=0, si es postivo el ajuste es 0

#PASO 3: CONFIGURACIÓN DE VARIABLES 
# col_2017, col_2018, nombre, color_17, color_18
variables = [
    ('ACT_2017_LOG', 'ACT_2018_LOG', 'Activos',    '#58A6FF', '#1F6FEB'),
    ('PAS_2017_LOG', 'PAS_2018_LOG', 'Pasivos',    '#F78166', '#B22222'),
    ('ING_2017_LOG', 'ING_2018_LOG', 'Ingresos',   '#3FB950', '#196C2E'),
    ('GAN_2017_LOG', 'GAN_2018_LOG', 'Ganancia',   '#D2A8FF', '#8957E5'),
    ('PAT_2017_LOG', 'PAT_2018_LOG', 'Patrimonio', '#FFA657', '#D4691E'),
]
#configuramos las tuplas asignando cada valor columna pero los datos logaritmicos, copiamos los nombres para evitar confusion por el cambio de nombre en agg

#PASO 4: CONSTRUCCIÓN DEL GRÁFICO
fig=go.Figure()
for col_17, col_18, nombre, color_17, color_18 in variables:

    # Línea 2017 — punteada
    fig.add_trace(go.Scatter(
        x             = sectores,
        y             = totales[col_17],
        name          = f"{nombre} 2017",
        mode          = 'lines+markers',
        line          = dict(color=color_17, width=2, dash='dot'),
        marker        = dict(size=8, color=color_17,
                             line=dict(width=1.5, color='white')),
        legendgroup   = nombre,
        hovertemplate = (
            f"<b>{nombre} 2017</b><br>"
            "Sector: %{x}<br>"
            "log₁₀(Total): %{y:.3f}<br>"
            "<extra></extra>"
        )
    ))
    #recorrido de tuplas definido en paso 3, definimos tipo de grafico,x=sectores, y=valores de las variables, nombre con year, line+markers lineas y puntos
    #siguen los mismo parametros de configueracion que se generaron en el boxplot, linea y punto
    # Línea 2018 — sólida
    fig.add_trace(go.Scatter(
        x             = sectores,
        y             = totales[col_18],
        name          = f"{nombre} 2018",
        mode          = 'lines+markers',
        line          = dict(color=color_18, width=2.5, dash='solid'),
        marker        = dict(size=8, color=color_18,
                             line=dict(width=1.5, color='white')),
        legendgroup   = nombre,
        hovertemplate = (
            f"<b>{nombre} 2018</b><br>"
            "Sector: %{x}<br>"
            "log₁₀(Total): %{y:.3f}<br>"
            "<extra></extra>"
        )
    ))
#lo mismo del year 2017 pero en 2018


# ── PASO 5: DISEÑO VISUAL 
#la configuracion del grafico es similar al del boxplot solo hay unos comentarios de algo nuevo en el codigo
fig.update_layout(
    title = dict(
        text  = "<b>TOTAL POR SECTOR ECONÓMICO — 2017 vs 2018</b><br>"
                "<sub>Activos, Pasivos, Ingresos, Ganancia y Patrimonio en escala log₁₀</sub>",
        font  = dict(size=18, color='white'),
        x     = 0.5
    ),
    paper_bgcolor = '#0D1117',
    plot_bgcolor  = '#161B22',
    font          = dict(color='#E6EDF3', size=12),
    xaxis         = dict(
        gridcolor = '#30363D',
        title     = 'Sector Económico',
        tickangle = -20
    ), #trickangle es para rotar los nombres y evitar solapamientos
    yaxis         = dict(
        gridcolor     = '#30363D',
        zerolinecolor = '#30363D',
        title         = 'log₁₀(Total COP)'
    ),
    legend = dict(
        bgcolor     = '#161B22',
        bordercolor = '#30363D',
        borderwidth = 1,
        groupclick  = 'toggleitem' #toggleitem muestra/aculta cada linea, togglegroup lo hace pero para todo el grupo
    ),
    height    = 650,
    hovermode = 'x unified',
    #hovermode para ver todos los valores de cada linea grfica no por punto independiente
    annotations = [dict(
        text    = "💡 Línea punteada = 2017 | Línea sólida = 2018 | Clic en leyenda para ocultar variables",
        xref="paper", yref="paper", x=0.5, y=-0.25,
        showarrow=False, font=dict(size=11, color='#8B949E'), xanchor='center'
    )]
)

fig.show()



Analisis de correlacion, solo se toman los datos 2018 ya que el comportamiento de los datos entre años es similar entonces el 2018 es mas que suficiente para evidenciar la correlacion entre las variables; se usa la correlacion de spearman ya que los datos obedecen a una distribucion normal.

In [22]:
from scipy import stats

#VARIABLES A ANALIZAR 
cols_2018 = ['ING_OP_2018','GANANCIA_2018','ACT_2018','PAS_2018','PAT_2018']

#MATRIZ DE CORRELACIÓN SPEARMAN
print("\n" + "="*65)
print("PASO 6 · CORRELACIÓN SPEARMAN 2018")
print("="*65)

corr_spearman = df[cols_2018].corr(method='spearman')

print("\n--- MATRIZ DE CORRELACIÓN ---")
print(corr_spearman.round(4))

#P-VALUES POR PAR
print("\n--- SIGNIFICANCIA ESTADÍSTICA POR PAR ---")
print(f"  {'Par de variables':45s}  {'r':>8}  {'p-value':>12}  {'Resultado'}")

pares = []
for i, col1 in enumerate(cols_2018):
    for j, col2 in enumerate(cols_2018):
        if j > i:
            r, p   = stats.spearmanr(df[col1], df[col2])
            sig    = "significativa" if p < 0.05 else "no significativa"
            pares.append((col1, col2, r, p))
            print(f"  {col1+' vs '+col2:45s}  {r:>8.4f}  {p:>12.2e}  {sig}")

#RANKING DE CORRELACIONES
print("\n--- RANKING DE CORRELACIONES ---")
pares.sort(key=lambda x: -abs(x[2]))
print(f"  {'Posición':>8}  {'Par':45s}  {'r':>8}  {'Fuerza'}")
for i, (col1, col2, r, p) in enumerate(pares):
    fuerza = "Muy fuerte" if abs(r) > 0.8 else \
             "Fuerte"     if abs(r) > 0.6 else \
             "Moderada"   if abs(r) > 0.4 else "Débil"
    print(f"  {i+1:>8}  {col1+' vs '+col2:45s}  {r:>8.4f}  {fuerza}")


PASO 6 · CORRELACIÓN SPEARMAN 2018

--- MATRIZ DE CORRELACIÓN ---
               ING_OP_2018  GANANCIA_2018  ACT_2018  PAS_2018  PAT_2018
ING_OP_2018         1.0000         0.3077    0.6670    0.6880    0.4760
GANANCIA_2018       0.3077         1.0000    0.4485    0.2015    0.5903
ACT_2018            0.6670         0.4485    1.0000    0.8398    0.8417
PAS_2018            0.6880         0.2015    0.8398    1.0000    0.5219
PAT_2018            0.4760         0.5903    0.8417    0.5219    1.0000

--- SIGNIFICANCIA ESTADÍSTICA POR PAR ---
  Par de variables                                      r       p-value  Resultado
  ING_OP_2018 vs GANANCIA_2018                     0.3077      2.23e-23  significativa
  ING_OP_2018 vs ACT_2018                          0.6670     1.08e-129  significativa
  ING_OP_2018 vs PAS_2018                          0.6880     3.94e-141  significativa
  ING_OP_2018 vs PAT_2018                          0.4760      1.12e-57  significativa
  GANANCIA_2018 vs ACT_2018

Podemos decir que: Las variables que mas correlacion tienen son los activos y el patrimonio, de igual forma los activos y los pasivos, dado que los valores de p-value son cercanos a 0 nos asegura que los valores generados de esta correlacion no son resultados del azar; como anotacion se infiere y resalta que estos resultados no quiere decir que una variable explique la otra, la corrrelacion hay que recordar que no es igual a causalidad, es decir para este caso el tener mas pasivos no es igual a tener mas activos empresariales.

In [23]:
#MATRIZ DE COVARIANZA
print("\n" + "="*65)
print("PASO 6B · COVARIANZA 2018")
print("="*65)

cols_2018 = ['ING_OP_2018','GANANCIA_2018','ACT_2018','PAS_2018','PAT_2018']

cov = df[cols_2018].cov()

print("\n--- MATRIZ DE COVARIANZA ---")
print(cov.map(lambda x: f"{x:,.0f}"))

#RANKING DE PARES
print("\n--- RANKING DE COVARIANZA ---")
print(f"  {'Par':45s}  {'Covarianza':>25}  {'r Spearman':>12}  {'Dirección'}")

pares_cov = []
for i, col1 in enumerate(cols_2018):
    for j, col2 in enumerate(cols_2018):
        if j > i:
            cov_val = cov.loc[col1, col2]
            r, _    = stats.spearmanr(df[col1], df[col2])
            pares_cov.append((col1, col2, cov_val, r))

pares_cov.sort(key=lambda x: -abs(x[2]))
for col1, col2, val, r in pares_cov:
    direccion = "↑ misma" if val > 0 else "↓ opuesta"
    print(f"  {col1+' vs '+col2:45s}  {val:>25,.0f}  {r:>12.4f}  {direccion}")

#VARIANZA POR VARIABLE
print("\n--- VARIANZA POR VARIABLE ---")
print(f"  {'Variable':25s}  {'Varianza':>30}  {'Interpretación'}")
varianzas = [
    ('ACT_2018',      'Activos',    cov.loc['ACT_2018','ACT_2018']),
    ('PAS_2018',      'Pasivos',    cov.loc['PAS_2018','PAS_2018']),
    ('PAT_2018',      'Patrimonio', cov.loc['PAT_2018','PAT_2018']),
    ('ING_OP_2018',   'Ingresos',   cov.loc['ING_OP_2018','ING_OP_2018']),
    ('GANANCIA_2018', 'Ganancia',   cov.loc['GANANCIA_2018','GANANCIA_2018']),
]
varianzas.sort(key=lambda x: -x[2])
for _, nombre, val in varianzas:
    nivel = "Alta dispersión" if val > 5e18 else \
            "Media dispersión" if val > 1e18 else "Baja dispersión"
    print(f"  {nombre:25s}  {val:>30,.0f}  {nivel}")


PASO 6B · COVARIANZA 2018

--- MATRIZ DE COVARIANZA ---
                             ING_OP_2018              GANANCIA_2018  \
ING_OP_2018    5,172,660,004,375,469,056    828,694,737,258,567,296   
GANANCIA_2018    828,694,737,258,567,296    202,392,795,929,853,120   
ACT_2018       9,020,875,461,870,653,440  1,751,009,699,214,581,248   
PAS_2018       4,459,062,437,588,429,824    788,293,248,546,932,608   
PAT_2018       4,561,813,024,282,267,648    962,716,450,667,651,072   

                                 ACT_2018                   PAS_2018  \
ING_OP_2018     9,020,875,461,870,653,440  4,459,062,437,588,429,824   
GANANCIA_2018   1,751,009,699,214,581,248    788,293,248,546,932,608   
ACT_2018       20,295,152,712,741,879,808  9,254,058,837,648,599,040   
PAS_2018        9,254,058,837,648,599,040  4,641,556,498,027,146,240   
PAT_2018       11,041,093,875,093,295,104  4,612,502,339,621,464,064   

                                 PAT_2018  
ING_OP_2018     4,561,813,024,282,267,6

Aca podemos visualizar que las empresas colombianas estan con una dispersin significativamente alta, ya que la covarianza entre las mismas variables es muy alta, por ende hay empresas con activos muy altos y empresas con activos demasiado bajos en relacion. Tambien podemos inferir que los activos y el patrimonio estan fuertemente relacionados, al deesviarse los activos de su promedio, el patrimonio tambien tiende a alejarse de su promedio; nos permite decir que las empresas grandes colombianas estan capitalizadas por activos propios.

#**Prueba de Hipotesis** En esta parte buscamos definir si las empresas tuvieron un crecimiento entre el 2017 y 2018 en colombia, para ello usamos la prueba wilcoxon, ya que esta prueba se aplica a datos no normales, y se esta evaluando las mismas variables en diferentes momentos de tiempo.

In [25]:
from scipy import stats
from itertools import combinations

print("\n" + "="*65)
print("PASO 7 · PRUEBAS DE HIPÓTESIS")
print("="*65)

# Variables más relevantes del análisis de correlación
variables_prueba = [
    ('ACT_2017',    'ACT_2018',    'Activos'),
    ('PAT_2017',    'PAT_2018',    'Patrimonio'),
    ('ING_OP_2017', 'ING_OP_2018', 'Ingresos'),
]

#PRUEBA 1: WILCOXON 2017 vs 2018
# H0: no hubo crecimiento entre 2017 y 2018
# H1: sí hubo crecimiento significativo
# Usamos Wilcoxon porque son las mismas empresas en dos momentos
print("\n--- WILCOXON — ¿Crecieron las empresas entre 2017 y 2018? ---")
print(f"  {'Variable':15s}  {'W':>12}  {'p-value':>12}  {'Mediana cambio':>20}  {'Resultado'}")

for col_17, col_18, nombre in variables_prueba:
    diferencia = df[col_18] - df[col_17]
    w, p       = stats.wilcoxon(diferencia)
    crecio     = diferencia.median() > 0
    resultado  = "creció significativamente" if crecio and p < 0.05 else "❌ no creció"
    print(f"  {nombre:15s}  {w:>12.0f}  {p:>12.2e}  "
          f"{diferencia.median():>20,.0f}  {resultado}")


PASO 7 · PRUEBAS DE HIPÓTESIS

--- WILCOXON — ¿Crecieron las empresas entre 2017 y 2018? ---
  Variable                    W       p-value        Mediana cambio  Resultado
  Activos                 84011      9.01e-74            17,569,438  creció significativamente
  Patrimonio             131916      3.57e-38             4,852,413  creció significativamente
  Ingresos                84091      6.40e-74            24,304,148  creció significativamente


**Interpretacion** Las empresas mas grandes de Colombia si tuvieron un crecimiento significativo entre 2017 y 2018

#**Prueba de Hipotesis** En esta parte buscamos definir si las empresas tuvieron la misma distribucion entre ellos; usamos la prueba de kruskal-wallis porque estamos comparando mas de 2 grupos que abedecen a una distribucion no normal.

In [26]:
#PRUEBA 2: KRUSKAL-WALLIS POR SECTOR
# H0: todos los sectores tienen la misma distribución
# H1: al menos un sector es significativamente diferente

print("\n--- KRUSKAL-WALLIS — ¿Los sectores son diferentes entre sí? ---")
print(f"  {'Variable':15s}  {'H':>10}  {'p-value':>12}  {'Resultado'}")

sectores = df['MACROSECTOR'].unique()
for col_18, nombre in [('ACT_2018','Activos'),
                        ('PAT_2018','Patrimonio'),
                        ('ING_OP_2018','Ingresos')]:
    grupos = [df[df['MACROSECTOR']==s][col_18].values for s in sectores]
    H, p   = stats.kruskal(*grupos)
    resultado = "diferencias significativas" if p < 0.05 else "sin diferencias"
    print(f"  {nombre:15s}  {H:>10.2f}  {p:>12.2e}  {resultado}")


--- KRUSKAL-WALLIS — ¿Los sectores son diferentes entre sí? ---
  Variable                  H       p-value  Resultado
  Activos              113.29      8.27e-23  diferencias significativas
  Patrimonio           135.35      1.74e-27  diferencias significativas
  Ingresos              17.13      4.26e-03  diferencias significativas


**Como resultado de la prueba de hipotesis kruskal-wallis** Segun los resultados obtenido rechazamos H0, de esto se puede decir:
1. Los mercados colombianos  no se comportan de igual forma a nivel empresarial ya que los ingresos, activos y patrimonio son divergentes entre ellos con gran diferencia
2. en cuanto a los ingresos que se perciben con menos diferencia nos indica quue la brecha entre los ingresos de las empresas es menor en relacion a los activos y patrimonio, eso quiere decir que las empresas colombianas no necesariamente tienen que tener ingresos superiores en el mercado para obtener mayor patrimonio y activos en relacion a las demas.

**ANALISIS POR PARES** Con los resultados obtenidos hast ahora se ha determinado que si hay diferencia entre los sectores economicos teniendo en cuenta sus activos, ingresos, pasivos y capital, pero no sabemos cuales son los sectores economicos que difieren entre si, entonces vamos a ahcer un analisis por pares para determinar cuales son los sectores que difieren entre si. Para ello vamos a realizar la prueba de evaluacion de mann-whitney y post-hoc.

In [28]:
from itertools import combinations
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

In [31]:
print("\n" + "="*65)
print("PASO 8 · ANÁLISIS POST-HOC — MANN-WHITNEY ENTRE PARES DE SECTORES")
print("="*65)
#usamos las 3 variables que resultaron significativas en la prueba Kruskal-Wallis
variables_posthoc=[('ACT_2018','Activos'),('PAT_2018','Patrimonio'),('ING_OP_2018','Ingresos')]
sectores_lista=df['MACROSECTOR'].unique().tolist()
pares=list(combinations(sectores_lista,2))
#Correlacion de Bonferroni: dividimos alpha entre el numero de comparaciones para no acumular error tipo I al hacer muchas comparciones simultaneas
n_comparaciones=len(pares)
alpha_original=0.05
alpha_bonf=alpha_original/n_comparaciones
print(f"\n  Sectores encontrados : {len(sectores_lista)}")
print(f"  Pares a comparar     : {n_comparaciones}")
print(f"  Alpha original       : {alpha_original}")
print(f"  Alpha Bonferroni     : {alpha_bonf:.5f}  (0.05 / {n_comparaciones})")
print(f"\n  → Solo se considera diferencia REAL si p < {alpha_bonf:.5f}")

#**prueba mann-whitney por variable**
#guardamos los datos en un diccionario para graficar despues
resultados_posthoc={} #{variable:dataframe con resultados}

for col, nombre in variables_posthoc:
    print(f"\n{'─'*65}")
    print(f"  {nombre.upper()} — Mann-Whitney por par de sectores")
    print(f"{'─'*65}")
    print(f"  {'Sector A':25s}  {'Sector B':25s}  {'U':>10}  {'p-value':>12}  {'Sig?':>6}  {'Mayor mediana'}")

    filas = []
    for s1, s2 in pares:
        g1    = df[df['MACROSECTOR'] == s1][col].values
        g2    = df[df['MACROSECTOR'] == s2][col].values

        # Mann-Whitney U — compara dos grupos independientes no normales
        U, p  = stats.mannwhitneyu(g1, g2, alternative='two-sided')

        # ¿Cuál sector tiene mayor mediana? — la dirección del efecto
        med1  = np.median(g1)
        med2  = np.median(g2)
        mayor = s1 if med1 > med2 else s2

        sig   = "1" if p < alpha_bonf else "0"
        filas.append({
            'sector_a' : s1,
            'sector_b' : s2,
            'U'        : U,
            'p_value'  : p,
            'sig'      : p < alpha_bonf,
            'mayor'    : mayor,
            'med_a'    : med1,
            'med_b'    : med2,
        })
        print(f"  {s1:25s}  {s2:25s}  {U:>10.0f}  {p:>12.2e}  {sig:>6}  {mayor}")

    resultados_posthoc[col] = filas

# ── RESUMEN EJECUTIVO ────────────────────────────────────────────
print("\n" + "="*65)
print("  RESUMEN EJECUTIVO POST-HOC")
print("="*65)

for col, nombre in variables_posthoc:
    filas     = resultados_posthoc[col]
    sig_count = sum(1 for f in filas if f['sig'])
    print(f"\n  {nombre}: {sig_count} de {n_comparaciones} pares con diferencia real (p < {alpha_bonf:.5f})")

    # Top 3 pares más distintos (p-value más bajo)
    top3 = sorted(filas, key=lambda x: x['p_value'])[:3]
    for f in top3:
        print(f"    → {f['sector_a']} vs {f['sector_b']}  |  p={f['p_value']:.2e}  |  más alto: {f['mayor']}")


PASO 8 · ANÁLISIS POST-HOC — MANN-WHITNEY ENTRE PARES DE SECTORES

  Sectores encontrados : 6
  Pares a comparar     : 15
  Alpha original       : 0.05
  Alpha Bonferroni     : 0.00333  (0.05 / 15)

  → Solo se considera diferencia REAL si p < 0.00333

─────────────────────────────────────────────────────────────────
  ACTIVOS — Mann-Whitney por par de sectores
─────────────────────────────────────────────────────────────────
  Sector A                   Sector B                            U       p-value    Sig?  Mayor mediana
  MINERO-HIDROCARBUROS       COMERCIO                        13722      2.19e-15       1  MINERO-HIDROCARBUROS
  MINERO-HIDROCARBUROS       MANUFACTURA                     12957      4.93e-08       1  MINERO-HIDROCARBUROS
  MINERO-HIDROCARBUROS       SERVICIOS                        8320      2.49e-05       1  MINERO-HIDROCARBUROS
  MINERO-HIDROCARBUROS       AGROPECUARIO                     1093      1.49e-06       1  MINERO-HIDROCARBUROS
  MINERO-HIDROCARBURO

**NOTA** el umbral de alpha se ajusto con boferroni ya que al tener un gran volumen de comparaciones se evitar tener errores productos del azar por la acumulacion de pruebas, umbral 0.00333

Dados los resultados obtenido las conclusiones se van a dar por cada variable evaluada en la prueba (activos, patrimonio e ingresos)
Activos: El sector de mineria e hidrocarburos tiene gran diferencia con respecto a los demas sectores economicos del pais, a excepcion del sector de la construccion donde no hay evidencia estadistica para decir que hay diferencia entre los activos de estos dos sectores, reforzando la hipotesis de que el sector de hidrocarburos liderado por la empresa ecopetrol, es el sector mas fuerte en colombia.
Por otra parte el sector del comercio es el sector mas debil en activos lo cual quiere decir que su operacion a pesar de que esta dada por altos margenes de rotacion, operan con poca infraestructura propia.
Patrimonio: Aca enconttramos una gran diferencia entre manufactura y comercio lo que nos indica que el sectormanufaturero colombiano opera en gran medida con infraestructura propia y capital de si mismo; por otra parte el sector minero en esta variable si es superior al sector de la construccion, donde su patrimonio es superior, esto quiere decir que aunque los activos de ambos sectores son similares y dominan el mercado colombiano, la construccion esta financiada por activos externos a diferencia del mineria e hidrocarburtos que si capital es propio.
Ingresos: Solo el sector de la mirenia e hidorcarburo tiene una diferencia notoria con respecto a los demas sectores, estos indica que los ingresos empresariales en los diferentes sectores economicos es similar entre si, esto nos confirma que a pesar de tener ingresos similares la riqueza colombiana esta mas concentrada en algunos sectores sobre la actividad comercial.